In [13]:
import io
import time
import struct
import socket
import threading
import numpy as np
import cv2
import av
import ipywidgets as widgets
from IPython.display import display
from pynq import Overlay, allocate, PL

PL.reset()
ol = Overlay("depack_pipeline_batch/depack_pipeline_batch.bit")

# DMA Channel Setup: axi_dma_0 (MM2S & LiDAR S2MM), axi_dma_1 (Camera S2MM)
dma_lidar = ol.axi_dma_0
dma_camera = ol.axi_dma_1

print(f"[INFO] dma_lidar.sendchannel max transfer  : {dma_lidar.sendchannel._max_size:,} bytes")
print(f"[INFO] dma_lidar.recvchannel max transfer  : {dma_lidar.recvchannel._max_size:,} bytes")
print(f"[INFO] dma_camera.recvchannel max transfer : {dma_camera.recvchannel._max_size:,} bytes")

MAX_PAYLOAD_BYTES = 1400
NUM_SLOTS = 16
LIDAR_SLOT_SIZE = 1024 * MAX_PAYLOAD_BYTES
TOTAL_LIDAR_BYTES = NUM_SLOTS * LIDAR_SLOT_SIZE
CAMERA_SLOT_SIZE = 1024 * MAX_PAYLOAD_BYTES
TOTAL_CAMERA_BYTES = NUM_SLOTS * CAMERA_SLOT_SIZE

buf_l2_cam = allocate(shape=(TOTAL_CAMERA_BYTES,), dtype=np.uint8, cacheable=False)
buf_l2_lidar = allocate(shape=(TOTAL_LIDAR_BYTES,), dtype=np.uint8, cacheable=False)
buf_l2_cam[:] = 0
buf_l2_lidar[:] = 0

phys_cam = buf_l2_cam.physical_address
phys_lidar = buf_l2_lidar.physical_address

reg_l2 = ol.layer2_depack_0.register_map
ol.layer2_depack_0.write(reg_l2.ddr_mem_camera_1.address, phys_cam & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_camera_2.address, (phys_cam >> 32) & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_lidar_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer2_depack_0.write(reg_l2.ddr_mem_lidar_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer2_depack_0.write(0x00, 0x81)

reg_l1_cam = ol.layer1_camera_prep_0.register_map
ol.layer1_camera_prep_0.write(reg_l1_cam.ddr_mem_1.address, phys_cam & 0xFFFFFFFF)
ol.layer1_camera_prep_0.write(reg_l1_cam.ddr_mem_2.address, (phys_cam >> 32) & 0xFFFFFFFF)
ol.layer1_camera_prep_0.write(0x00, 0x81)

reg_l1 = ol.layer1_depack_0.register_map
ol.layer1_depack_0.write(reg_l1.ddr_mem_meta_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_meta_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_points_1.address, phys_lidar & 0xFFFFFFFF)
ol.layer1_depack_0.write(reg_l1.ddr_mem_points_2.address, (phys_lidar >> 32) & 0xFFFFFFFF)
ol.layer1_depack_0.write(0x00, 0x81)

ol.packet_dispatcher_0.write(0x00, 0x81)

CAMERA_DESC_WORDS = 9
MAX_WORDS_PER_LIDAR_FRAME = (LIDAR_SLOT_SIZE + 3) // 4
MAX_WORDS_PER_CAMERA_FRAME = (CAMERA_SLOT_SIZE + 3) // 4 + CAMERA_DESC_WORDS
INACTIVITY_TIMEOUT_SEC = 3.0
LISTEN_PORT = 9000
STATUS_PRINT_EVERY = 30

BATCH_PACKET_COUNT = 32
BATCH_FLUSH_TIMEOUT_SEC = 0.005

PIXEL_FORMAT_NAMES = {0: "RGB", 1: "BGR", 2: "RGBA", 3: "GRAY"}

[INFO] dma_lidar.sendchannel max transfer  : 67,108,863 bytes
[INFO] dma_lidar.recvchannel max transfer  : 67,108,863 bytes
[INFO] dma_camera.recvchannel max transfer : 67,108,863 bytes


In [14]:
def dma_channel_error(channel):
    try:
        return bool(channel.error)
    except Exception:
        return None

def wait_for_idle(channel, timeout_s, label):
    t0 = time.time()
    while not channel.idle:
        if dma_channel_error(channel):
            print(f"[ERROR] {label}: DMA reported an error state.")
            return False
        if time.time() - t0 > timeout_s:
            print(f"[TIMEOUT] {label} did not go idle within {timeout_s}s.")
            return False
        time.sleep(0.0005)
    return True

def unpack_camera_field_desc(words9):
    raw = struct.pack("<9I", *[int(w) for w in words9])
    slot_index = raw[0]
    sensor_id = raw[1]
    frame_id = struct.unpack("<I", raw[2:6])[0]
    valid = raw[6]
    width = struct.unpack("<H", raw[7:9])[0]
    height = struct.unpack("<H", raw[9:11])[0]
    gop_size = raw[11]
    crf = raw[12]
    shared_field = raw[13]

    ts_bytes = raw[14:22]
    try:
        timestamp = struct.unpack(">d", ts_bytes)[0]
    except Exception:
        timestamp = 0.0

    bitstream_addr = struct.unpack("<Q", raw[22:30])[0]
    bitstream_length = struct.unpack("<I", raw[30:34])[0]
    compressed = bool(raw[34] & 0x01)

    profile_id = shared_field if compressed else None
    pixel_format = None if compressed else shared_field

    return {
        "slot_index": slot_index,
        "sensor_id": sensor_id,
        "frame_id": frame_id,
        "valid": valid,
        "compressed": compressed,
        "width": width,
        "height": height,
        "gop": gop_size,
        "crf": crf,
        "profile_id": profile_id,
        "pixel_format": pixel_format,
        "timestamp": timestamp,
        "bitstream_addr": bitstream_addr,
        "bitstream_length": bitstream_length,
    }

h264_codec = av.CodecContext.create('h264', 'r')

def decode_camera_frame_to_bgr(fd, raw_bytes):
    w, h = fd["width"], fd["height"]

    if fd["compressed"]:
        try:
            packet = av.Packet(raw_bytes)
            frames = h264_codec.decode(packet)
            if frames:
                return frames[0].to_ndarray(format="bgr24")
        except Exception:
            return None

    if w <= 0 or h <= 0:
        return None

    pix_fmt = fd["pixel_format"]
    if pix_fmt == 0 and len(raw_bytes) == w * h * 3:       # RGB
        raw = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((h, w, 3))
        return cv2.cvtColor(raw, cv2.COLOR_RGB2BGR)
    elif pix_fmt == 1 and len(raw_bytes) == w * h * 3:     # BGR
        return np.frombuffer(raw_bytes, dtype=np.uint8).reshape((h, w, 3)).copy()
    elif pix_fmt == 2 and len(raw_bytes) == w * h * 4:     # RGBA
        raw = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((h, w, 4))
        return cv2.cvtColor(raw, cv2.COLOR_RGBA2BGR)
    elif pix_fmt == 3 and len(raw_bytes) == w * h:         # GRAY
        raw = np.frombuffer(raw_bytes, dtype=np.uint8).reshape((h, w))
        return cv2.cvtColor(raw, cv2.COLOR_GRAY2BGR)

    return None

def decode_lidar_frame(raw_buf):
    def swap32(word):
        word = int(word)
        return (((word & 0xFF) << 24) | (((word >> 8) & 0xFF) << 16) |
                (((word >> 16) & 0xFF) << 8) | ((word >> 24) & 0xFF))

    if len(raw_buf) < 5 or int(raw_buf[0]) == 0:
        return None

    frame_id = swap32(raw_buf[0])
    word1 = int(raw_buf[1])
    sensor_id = word1 & 0xFF
    compressed = bool((word1 >> 8) & 0xFF)
    num_points = swap32(raw_buf[2])

    w3, w4 = int(raw_buf[3]), int(raw_buf[4])
    ts_bytes = bytes([w3 & 0xFF, (w3 >> 8) & 0xFF, (w3 >> 16) & 0xFF, (w3 >> 24) & 0xFF,
                      w4 & 0xFF, (w4 >> 8) & 0xFF, (w4 >> 16) & 0xFF, (w4 >> 24) & 0xFF])
    try:
        timestamp = struct.unpack(">d", ts_bytes)[0]
    except Exception:
        timestamp = 0.0

    max_points_in_buf = (len(raw_buf) - 5) // 2
    n = min(int(num_points), max_points_in_buf)
    if n <= 0:
        return {"frame_id": frame_id, "sensor_id": sensor_id, "compressed": compressed,
                "timestamp": timestamp, "points_xyzi": np.zeros((0, 4), dtype=np.float32)}

    wordA = raw_buf[5:5 + 2 * n:2].astype(np.uint32)
    wordB = raw_buf[6:6 + 2 * n:2].astype(np.uint32)

    distance_raw = ((wordA & 0xFF) << 8) | ((wordA >> 8) & 0xFF)
    azimuth_raw = (((wordA >> 16) & 0xFF) << 8) | ((wordA >> 24) & 0xFF)
    el_bits = (((wordB & 0xFF) << 8) | ((wordB >> 8) & 0xFF)).astype(np.uint16)
    elevation_raw = el_bits.astype(np.int16)
    reflectivity_raw = ((wordB >> 16) & 0xFF).astype(np.uint8)

    r = distance_raw.astype(np.float64) * 0.05
    az = np.deg2rad(azimuth_raw.astype(np.float64) * 0.1)
    el = np.deg2rad(elevation_raw.astype(np.float64) * 0.1)

    x = r * np.cos(el) * np.sin(az)
    y = r * np.cos(el) * np.cos(az)
    z = r * np.sin(el)
    intensity = reflectivity_raw.astype(np.float32) / 255.0

    points_xyzi = np.stack([x, y, z, intensity], axis=1).astype(np.float32)
    return {"frame_id": frame_id, "sensor_id": sensor_id, "compressed": compressed,
            "timestamp": timestamp, "points_xyzi": points_xyzi}

def render_bev(points_xyzi, size=480, range_m=120.0):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    if points_xyzi.shape[0] == 0:
        return img

    valid_finite = np.isfinite(points_xyzi).all(axis=1)
    points_xyzi = points_xyzi[valid_finite]
    if points_xyzi.shape[0] == 0:
        return img

    x, y, intensity = points_xyzi[:, 0], points_xyzi[:, 1], points_xyzi[:, 3]
    px = ((x / range_m) * (size / 2) + size / 2).astype(np.int32)
    py = (size / 2 - (y / range_m) * (size / 2)).astype(np.int32)

    valid = (px >= 0) & (px < size) & (py >= 0) & (py < size)
    px, py, intensity = px[valid], py[valid], intensity[valid]

    shade = np.clip(intensity * 200 + 55, 0, 255).astype(np.uint8)
    img[py, px, 1] = shade
    img[py, px, 2] = shade // 2
    cv2.drawMarker(img, (size // 2, size // 2), (0, 0, 255), cv2.MARKER_TRIANGLE_UP, 12, 2)
    return img

In [ ]:
rx_buf0 = allocate(shape=(MAX_WORDS_PER_LIDAR_FRAME,), dtype=np.uint32, cacheable=False)
rx_buf1 = allocate(shape=(MAX_WORDS_PER_LIDAR_FRAME,), dtype=np.uint32, cacheable=False)
rx_bufs = [rx_buf0, rx_buf1]

cam_rx_buf0 = allocate(shape=(MAX_WORDS_PER_CAMERA_FRAME,), dtype=np.uint32, cacheable=False)
cam_rx_buf1 = allocate(shape=(MAX_WORDS_PER_CAMERA_FRAME,), dtype=np.uint32, cacheable=False)
cam_rx_bufs = [cam_rx_buf0, cam_rx_buf1]

dma_lidar.sendchannel.start()
dma_lidar.recvchannel.start()
dma_camera.recvchannel.start()

rx_active = True
total_bytes_received = 0
first_packet_time = 0.0
last_packet_time = 0.0
send_stalled = False

active_rx = 0
active_cam_rx = 0

cam_frame_count = 0
cam_decoded_count = 0
cam_failed_count = 0
lidar_frame_count = 0
lidar_decoded_count = 0

cam_widget = widgets.Image(format='jpeg', width=640, height=320)
bev_widget = widgets.Image(format='jpeg', width=640, height=320)
status_label = widgets.Label("Waiting for CARLA stream...")

ui_box = widgets.VBox([
    status_label,
    widgets.HBox([
        widgets.VBox([widgets.Label("Camera Live Playback (H.264 Decoded)"), cam_widget]),
        widgets.VBox([widgets.Label("LiDAR Live BEV Playback"), bev_widget])
    ])
])
display(ui_box)

dma_lidar.recvchannel.transfer(rx_bufs[active_rx])
dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])

def udp_rx_dma_worker():
    global rx_active, total_bytes_received, first_packet_time, last_packet_time, send_stalled

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEPORT, 1)
    except (AttributeError, OSError):
        pass
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 32 * 1024 * 1024)

    try:
        sock.bind(("0.0.0.0", LISTEN_PORT))
    except OSError as e:
        print(f"[NET WORKER ERROR] bind() failed: {e}")
        sock.close()
        rx_active = False
        return

    print(f"[NET WORKER] Listening on UDP port {LISTEN_PORT}...")

    tx_dma_buf = allocate(shape=(BATCH_PACKET_COUNT * MAX_PAYLOAD_BYTES,),
                          dtype=np.uint8, cacheable=False)

    batch_chunks = []
    batch_bytes = 0
    batch_count = 0
    batch_deadline = None
    dma_in_flight = False
    last_packet_time = time.time()

    def flush_batch():
        global send_stalled
        nonlocal batch_chunks, batch_bytes, batch_count, batch_deadline, dma_in_flight
        if batch_count == 0:
            return

        if dma_in_flight and not wait_for_idle(dma_lidar.sendchannel, 2.0, "sendchannel"):
            send_stalled = True
            batch_chunks, batch_bytes, batch_count, batch_deadline = [], 0, 0, None
            return

        blob = b"".join(batch_chunks)
        blob_len = len(blob)
        if blob_len > tx_dma_buf.nbytes:
            print(f"[WARN] Batch size ({blob_len} B) exceeded buffer capacity ({tx_dma_buf.nbytes} B). Truncating.")
            blob_len = tx_dma_buf.nbytes

        tx_dma_buf[:blob_len] = np.frombuffer(blob[:blob_len], dtype=np.uint8)
        dma_lidar.sendchannel.transfer(tx_dma_buf[:blob_len])
        dma_in_flight = True

        batch_chunks, batch_bytes, batch_count, batch_deadline = [], 0, 0, None

    def add_to_batch(pkt_bytes):
        nonlocal batch_chunks, batch_bytes, batch_count, batch_deadline
        length = len(pkt_bytes)
        pad = (-length) % 4
        padded_size = length + pad

        if batch_bytes + padded_size > tx_dma_buf.nbytes:
            flush_batch()

        batch_chunks.append(pkt_bytes)
        if pad:
            batch_chunks.append(b'\x00' * pad)

        batch_bytes += padded_size
        batch_count += 1

        if batch_deadline is None:
            batch_deadline = time.time() + BATCH_FLUSH_TIMEOUT_SEC

    try:
        while rx_active:
            try:
                timeout = 0.5
                if batch_deadline is not None:
                    timeout = max(0.0001, min(timeout, batch_deadline - time.time()))
                sock.settimeout(timeout)

                data, addr = sock.recvfrom(2048)
                current_time = time.time()
                last_packet_time = current_time
                if first_packet_time == 0.0:
                    first_packet_time = current_time

                if data == b"EOF_STREAM":
                    flush_batch()
                    if dma_in_flight:
                        if not wait_for_idle(dma_lidar.sendchannel, 2.0, "sendchannel (EOF flush)"):
                            send_stalled = True
                    print("[NET WORKER] EOF Stream signal received.")
                    break

                total_bytes_received += len(data)
                add_to_batch(data)

                if batch_count >= BATCH_PACKET_COUNT:
                    flush_batch()

            except socket.timeout:
                if batch_deadline is not None and time.time() >= batch_deadline:
                    flush_batch()
                if first_packet_time > 0.0 and (time.time() - last_packet_time >= INACTIVITY_TIMEOUT_SEC):
                    print(f"[NET WORKER] No packets received for {INACTIVITY_TIMEOUT_SEC}s. Stopping stream.")
                    flush_batch()
                    break
                continue
            except Exception as e:
                print(f"[NET WORKER ERROR] {e}")
                break
    finally:
        try:
            flush_batch()
        except Exception as e:
            print(f"[NET WORKER] Final flush error: {e}")
        try:
            sock.close()
        except Exception:
            pass
        print("[NET WORKER] Network thread finished.")

def process_camera_completion():
    global active_cam_rx, cam_frame_count, cam_decoded_count, cam_failed_count
    if not dma_camera.recvchannel.idle:
        return

    curr_buf = cam_rx_bufs[active_cam_rx]
    if hasattr(curr_buf, "invalidate"):
        curr_buf.invalidate()

    fd = unpack_camera_field_desc(curr_buf[0:CAMERA_DESC_WORDS])
    bitstream_length = fd["bitstream_length"]

    if fd["valid"] and bitstream_length > 0:
        words_count = (bitstream_length + 3) // 4
        if CAMERA_DESC_WORDS + words_count <= len(curr_buf):
            raw_bytes = curr_buf[CAMERA_DESC_WORDS: CAMERA_DESC_WORDS + words_count].tobytes()[:bitstream_length]
            img_bgr = decode_camera_frame_to_bgr(fd, raw_bytes)
            if img_bgr is not None:
                cam_decoded_count += 1
                # Update widget immediately upon decode completion
                _, jpeg = cv2.imencode('.jpg', img_bgr)
                cam_widget.value = jpeg.tobytes()
            else:
                cam_failed_count += 1
        else:
            cam_failed_count += 1
        cam_frame_count += 1

    if cam_frame_count and cam_frame_count % STATUS_PRINT_EVERY == 0:
        print(f"[LIVE] Camera frames: {cam_frame_count} (decoded {cam_decoded_count}, failed {cam_failed_count})")

    active_cam_rx = 1 - active_cam_rx
    if rx_active:
        dma_camera.recvchannel.transfer(cam_rx_bufs[active_cam_rx])

def process_lidar_completion():
    global active_rx, lidar_frame_count, lidar_decoded_count
    if not dma_lidar.recvchannel.idle:
        return

    curr_buf = rx_bufs[active_rx]
    if hasattr(curr_buf, "invalidate"):
        curr_buf.invalidate()

    parsed = decode_lidar_frame(curr_buf)
    if parsed is not None:
        lidar_frame_count += 1
        if parsed["points_xyzi"].shape[0] > 0:
            lidar_decoded_count += 1
            # Update widget immediately upon decode completion
            bev_img = render_bev(parsed["points_xyzi"])
            _, jpeg = cv2.imencode('.jpg', bev_img)
            bev_widget.value = jpeg.tobytes()

        if lidar_frame_count % STATUS_PRINT_EVERY == 0:
            print(f"[LIVE] LiDAR frames: {lidar_frame_count} (rendered {lidar_decoded_count})")

    active_rx = 1 - active_rx
    if rx_active:
        dma_lidar.recvchannel.transfer(rx_bufs[active_rx])

def service_hardware():
    process_camera_completion()
    process_lidar_completion()

In [ ]:
# Live Stream Execution Loop & Summary
net_thread = threading.Thread(target=udp_rx_dma_worker, daemon=True)
net_thread.start()
status_label.value = (f"Streaming live on UDP port {LISTEN_PORT} (batched) - "
                      f"stops on EOF or {INACTIVITY_TIMEOUT_SEC}s silence")

while net_thread.is_alive():
    service_hardware()
    time.sleep(0.0001)

rx_active = False
status_label.value = "Stream ended - draining in-flight packets..."

t_drain = time.time()
while time.time() - t_drain < 1.0:
    service_hardware()
    time.sleep(0.0005)

net_thread.join()

total_exec_time = (last_packet_time - first_packet_time) if (first_packet_time and last_packet_time) else 0.0
status_label.value = (f"Stream finished - {cam_frame_count} camera / {lidar_frame_count} LiDAR frames "
                      f"in {total_exec_time:.3f}s")

print("\n" + "=" * 70)
print(" LIVE STREAM SUMMARY")
print("=" * 70)
print(f" Send Stalled          : {send_stalled}")
print(f" Total Stream Time     : {total_exec_time:.3f} s")
print(f" Total Bytes Received  : {total_bytes_received:,} B")
print(f" Camera Frames         : {cam_frame_count} received | {cam_decoded_count} decoded | {cam_failed_count} failed")
print(f" LiDAR Frames          : {lidar_frame_count} received | {lidar_decoded_count} rendered")
print("=" * 70)

[NET WORKER] Listening on UDP port 9000...
[LIVE] LiDAR frames: 30 (rendered 30)
[LIVE] Camera frames: 30 (decoded 30, failed 0)
[LIVE] LiDAR frames: 60 (rendered 60)
[LIVE] Camera frames: 60 (decoded 60, failed 0)
[LIVE] LiDAR frames: 90 (rendered 90)
[LIVE] Camera frames: 90 (decoded 90, failed 0)
[LIVE] LiDAR frames: 120 (rendered 120)
[LIVE] Camera frames: 120 (decoded 120, failed 0)
[LIVE] LiDAR frames: 150 (rendered 150)
[LIVE] Camera frames: 150 (decoded 150, failed 0)
[LIVE] LiDAR frames: 180 (rendered 180)
[LIVE] Camera frames: 180 (decoded 180, failed 0)
[LIVE] LiDAR frames: 210 (rendered 210)
[LIVE] Camera frames: 210 (decoded 210, failed 0)
[LIVE] LiDAR frames: 240 (rendered 240)
[LIVE] Camera frames: 240 (decoded 240, failed 0)
[LIVE] LiDAR frames: 270 (rendered 270)
[LIVE] Camera frames: 270 (decoded 270, failed 0)
[LIVE] LiDAR frames: 300 (rendered 300)
[LIVE] Camera frames: 300 (decoded 300, failed 0)
[LIVE] LiDAR frames: 330 (rendered 330)
[LIVE] Camera frames: 330 (de